In [1]:
# FASE 1: Conexión y arquitectura del Data Lake
from hdfs import InsecureClient

# Conexión al clúster HDFS
client = InsecureClient('http://localhost:9870', user='ubuntu')

# Estructura de directorios a crear
directorios = [
    '/datalake/renfe/raw/estaciones',
    '/datalake/renfe/raw/viajeros',
    '/datalake/renfe/processed',
    '/datalake/renfe/logs'
]

# Crear directorios con permisos 755
for directorio in directorios:
    client.makedirs(directorio, permission='755')
    print(f"✓ Creado: {directorio}")

# Verificar estructura creada
print("\n--- Verificación de /datalake/renfe ---")
for item in client.list('/datalake/renfe', status=True):
    nombre, info = item
    print(f"  {nombre} | tipo: {info['type']} | permisos: {info['permission']}")

✓ Creado: /datalake/renfe/raw/estaciones
✓ Creado: /datalake/renfe/raw/viajeros
✓ Creado: /datalake/renfe/processed
✓ Creado: /datalake/renfe/logs

--- Verificación de /datalake/renfe ---
  logs | tipo: DIRECTORY | permisos: 755
  processed | tipo: DIRECTORY | permisos: 755
  raw | tipo: DIRECTORY | permisos: 755


In [2]:
# FASE 2: Ingesta concurrente de datos crudos (Zona Raw)

# Subir estaciones_completo.csv y estaciones_feve.csv
archivos_estaciones = [
    ('/home/ubuntu/estaciones_completo.csv', '/datalake/renfe/raw/estaciones/estaciones_completo.csv'),
    ('/home/ubuntu/estaciones_feve.csv',     '/datalake/renfe/raw/estaciones/estaciones_feve.csv'),
]

for local, hdfs_path in archivos_estaciones:
    client.upload(hdfs_path, local, overwrite=True, n_threads=4)
    print(f"✓ Subido: {hdfs_path}")

# Subir viajeros_tramo.csv
client.upload(
    '/datalake/renfe/raw/viajeros/viajeros_tramo.csv',
    '/home/ubuntu/viajeros_tramo.csv',
    overwrite=True,
    n_threads=4
)
print("✓ Subido: /datalake/renfe/raw/viajeros/viajeros_tramo.csv")

# Auditoría de archivos en /raw/estaciones
print("\n--- Auditoría /datalake/renfe/raw/estaciones ---")
for nombre, info in client.list('/datalake/renfe/raw/estaciones', status=True):
    print(f"  Archivo:     {nombre}")
    print(f"  Tamaño:      {info['length']} bytes")
    print(f"  Permisos:    {info['permission']}")
    print(f"  Replicación: {info['replication']}")
    print()

✓ Subido: /datalake/renfe/raw/estaciones/estaciones_completo.csv
✓ Subido: /datalake/renfe/raw/estaciones/estaciones_feve.csv
✓ Subido: /datalake/renfe/raw/viajeros/viajeros_tramo.csv

--- Auditoría /datalake/renfe/raw/estaciones ---
  Archivo:     estaciones_completo.csv
  Tamaño:      142155 bytes
  Permisos:    644
  Replicación: 1

  Archivo:     estaciones_feve.csv
  Tamaño:      61795 bytes
  Permisos:    644
  Replicación: 1



In [3]:
import csv
import unicodedata

# FASE 3: Procesamiento en Streaming y filtrado (Zona Processed)
PROVINCIA_OBJETIVO = 'LEÓN'
IDX_PROVINCIA = 7  # Columna 'PROVINCIA' verificada en la cabecera
DELIMITADOR = ';'

ruta_raw = '/datalake/renfe/raw/estaciones/estaciones_completo.csv'
ruta_processed = '/datalake/renfe/processed/estaciones_filtradas.csv'


def normalizar_texto(texto: str) -> str:
    """Elimina tildes/diacríticos y normaliza a mayúsculas para evitar fallos entre 'LEÓN' y 'LEON'."""
    sin_tildes = ''.join(
        c for c in unicodedata.normalize('NFD', texto.strip().upper())
        if unicodedata.category(c) != 'Mn'
    )
    return sin_tildes


provincia_norm = normalizar_texto(PROVINCIA_OBJETIVO)
num_estaciones = 0

# Doble flujo en streaming: lee de /raw en latin-1 y escribe en /processed en utf-8 con O(1) en RAM
with client.read(ruta_raw, encoding='latin-1') as reader, \
     client.write(ruta_processed, encoding='utf-8', overwrite=True) as writer:

    csv_reader = csv.reader(reader, delimiter=DELIMITADOR)
    csv_writer = csv.writer(writer, delimiter=DELIMITADOR, quoting=csv.QUOTE_ALL, lineterminator='\n')

    # 1. Conservar cabecera
    cabecera = next(csv_reader, None)
    if cabecera is not None:
        csv_writer.writerow(cabecera)

        # 2. Filtrar línea a línea exclusivamente por la columna PROVINCIA
        for fila in csv_reader:
            if len(fila) > IDX_PROVINCIA and normalizar_texto(fila[IDX_PROVINCIA]) == provincia_norm:
                csv_writer.writerow(fila)
                num_estaciones += 1

print(f"✓ Estaciones encontradas para {PROVINCIA_OBJETIVO}: {num_estaciones}")
print(f"✓ Guardado en streaming en {ruta_processed}")

# 3. Verificación en streaming de las primeras 5 líneas del fichero resultante
print("\n--- Primeras 5 líneas del archivo filtrado ---")
with client.read(ruta_processed, encoding='utf-8') as reader:
    for i, linea in enumerate(reader):
        if i >= 5:
            break
        print(linea.strip())

✓ Estaciones encontradas para LEÓN: 43
✓ Guardado en streaming en /datalake/renfe/processed/estaciones_filtradas.csv

--- Primeras 5 líneas del archivo filtrado ---
"CODIGO";"DESCRIPCION";"LATITUD";"LONGITUD";"DIRECION";"CP";"POBLACION";"PROVINCIA";"PAIS";"CERCANIAS";"ANCHO MÉTRICO";"COMUN"
"05755";"LA LLAMA DE LA GUZPEÑA";"42.7702481";"-5.0412478";"Ctra. Cl - 626 --- Estación Ferrocarril, S/N";"24893";"PRADO DE LA GUZPEÑA";"LEÓN";"ESPAÑA";"SI";"SI";""
"05756";"LA ESPINA";"42.7868560";"-4.9052025";"Ctra. Cl - 626 --- Estación Ferrocarril, S/N";"24889";"VALDERRUEDA";"LEÓN";"ESPAÑA";"SI";"SI";""
"05757";"VALCUENDE";"42.7655780";"-4.9292558";"C/ Arriba, S/N";"24889";"VALDERRUEDA";"LEÓN";"ESPAÑA";"SI";"SI";""
"05758";"PUENTE ALMUHEY";"42.7876320";"-4.9718992";"Ctra. General  S/N- Puente Almuhey";"24880";"VALDERRUEDA";"LEÓN";"ESPAÑA";"SI";"SI";""


In [4]:
# FASE 4: Registro de auditoría (Logging con append)
ruta_log = '/datalake/renfe/logs/ingesta.log'

# 1. Primer evento: inicio del pipeline (sobrescribe log previo)
with client.write(ruta_log, encoding='utf-8', overwrite=True) as writer:
    writer.write('[INICIO] Pipeline ejecutado. Archivos crudos cargados en /raw.\n')
print("✓ Log de inicio escrito")

# 2. Segundo evento: resultado del filtrado (append al final del fichero)
with client.write(ruta_log, encoding='utf-8', append=True) as writer:
    writer.write(
        f'[PROCESADO] Se han filtrado un total de {num_estaciones} estaciones para la provincia seleccionada.\n'
    )
print("✓ Log de procesado añadido")

# 3. Lectura completa del log
print("\n--- Contenido de ingesta.log ---")
with client.read(ruta_log, encoding='utf-8') as reader:
    print(reader.read())

✓ Log de inicio escrito
✓ Log de procesado añadido

--- Contenido de ingesta.log ---
[INICIO] Pipeline ejecutado. Archivos crudos cargados en /raw.
[PROCESADO] Se han filtrado un total de 43 estaciones para la provincia seleccionada.



In [5]:
import os

# FASE 5: Exportación para el equipo de analítica y limpieza

ruta_hdfs_filtradas = '/datalake/renfe/processed/estaciones_filtradas.csv'
ruta_local_entrega = '/home/ubuntu/entrega_estaciones.csv'
ruta_hdfs_feve = '/datalake/renfe/raw/estaciones/estaciones_feve.csv'
dir_raw_estaciones = '/datalake/renfe/raw/estaciones'

# 1. Descargar fichero procesado a local con nombre específico
client.download(
    ruta_hdfs_filtradas,
    ruta_local_entrega,
    overwrite=True
)

tamano_local = os.path.getsize(ruta_local_entrega)
print(f"✓ Archivo descargado en local: {ruta_local_entrega} ({tamano_local} bytes)")

# 2. Política de retención: eliminar únicamente estaciones_feve.csv (recursive=False por seguridad)
eliminado = client.delete(ruta_hdfs_feve, recursive=False)
print(f"✓ Eliminado {ruta_hdfs_feve}: {eliminado}")

# Confirmar que estaciones_completo.csv permanece intacto
print(f"\n--- Contenido actual de {dir_raw_estaciones} ---")
for nombre, info in client.list(dir_raw_estaciones, status=True):
    print(f"  - {nombre} ({info['length']} bytes)")


# 3. Función de reinicio completo del Data Lake
def reset_datalake(hdfs_client=client, ruta_base='/datalake/renfe'):
    """Elimina de forma recursiva toda la estructura /datalake/renfe."""
    eliminado_total = hdfs_client.delete(ruta_base, recursive=True)
    if eliminado_total:
        print(f"✓ Data Lake ({ruta_base}) eliminado por completo.")
    else:
        print(f"⚠ La ruta {ruta_base} no existía o no pudo eliminarse.")


# IMPORTANTE: Mantener comentada para evitar borrar las evidencias en HDFS si se ejecuta "Run All"
# reset_datalake()

✓ Archivo descargado en local: /home/ubuntu/entrega_estaciones.csv (5921 bytes)
✓ Eliminado /datalake/renfe/raw/estaciones/estaciones_feve.csv: True

--- Contenido actual de /datalake/renfe/raw/estaciones ---
  - estaciones_completo.csv (142155 bytes)
